# AlphaLens Model Registry

This notebook audits the latest immutable model package. Registration stores the native model, ordered feature contract, data fingerprints, metrics, model card, reference predictions, and artifact checksums. Promotion is automatic and evidence-based; a rejected research model remains reproducible but cannot be served as champion.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.ml.registry import load_registered_model, verify_registered_model

REGISTRY = ROOT / 'data' / 'ml' / 'registry'

c:\Users\Lixing\Desktop\alphalens\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Registry index

In [2]:
index_path = REGISTRY / 'registry.json'
if not index_path.exists():
    raise FileNotFoundError('Run python -m pipelines.ml.registry first.')
index = json.loads(index_path.read_text(encoding='utf-8'))
display(pd.DataFrame(index['models']))
print('Latest:', index['latest_version'])
print('Champion:', index['champion_version'])

,created_at,manifest_sha256,status,version
0,2026-09-17T06:47:13.928299+00:00,f3ba001c6b16aa1c065e087e5a36e8f05b0a536dfff371...,rejected,xgboost-20260917T064713Z-7c7c72ec


Latest: xgboost-20260917T064713Z-7c7c72ec
Champion: None


## Integrity and reproducibility verification

In [3]:
version = index['latest_version']
version_directory = REGISTRY / version
manifest = json.loads((version_directory / 'manifest.json').read_text(encoding='utf-8'))
verification = verify_registered_model(version_directory)
display(pd.Series(verification, name='value'))
assert verification['passed']

passed                                                   True
version                     xgboost-20260917T064713Z-7c7c72ec
status                                               rejected
artifacts_checked                                           6
reference_rows                                              5
max_reference_difference                                  0.0
errors                                                     []
Name: value, dtype: object

In [4]:
artifacts = pd.DataFrame([
    {'artifact': name, **metadata}
    for name, metadata in manifest['artifacts'].items()
]).sort_values('artifact')
display(artifacts)

,artifact,bytes,file,sha256
0,feature_schema,12599,feature_schema.json,45edeb03cf7f1a4e85f42580b3411e24a60e9b1a3ec535...
1,metrics,12501,metrics.json,68636033d8ba58ceafc5e94c05119e9006ffbf3fb68740...
2,model,53611,model.json,dbd22b09792348e6635e3022d1d33e20e7c12133ef7802...
3,model_card,728,model_card.md,db763504deff2b1eaad792bbaacf824d2202730b96cb54...
4,reference_features,5614,reference_features.csv,abeb11333725d699ea358fd387a5a7488372c82261c5a1...
5,reference_predictions,495,reference_predictions.json,296d3c0c2c5f7953090442de0a4ef5f35878aa176a40b2...


## Promotion decision

In [5]:
promotion = manifest['promotion']
display(pd.DataFrame([
    {'gate': name, **details}
    for name, details in promotion['checks'].items()
]))
print('Status:', promotion['status'].upper())
for reason in promotion['reasons']:
    print('-', reason)

,gate,passed,rule,value,absolute_improvement,historical_mean_mae,model_mae
0,net_long_short_sharpe,False,sharpe > 0 after transaction costs,-0.407381,NaN,NaN,NaN
1,test_mae_improvement,False,model_mae < historical_mean_mae,NaN,-0.000358,0.06266,0.063018


Status: REJECTED
- Selected XGBoost test MAE did not beat historical_mean.
- Net long-short test Sharpe was not positive.


## Feature and data contract

In [6]:
schema = json.loads((version_directory / 'feature_schema.json').read_text(encoding='utf-8'))
display(pd.DataFrame(schema['features']))
display(pd.Series(manifest['dataset_fingerprints'], name='sha256'))
display(pd.Series(manifest['training_window'], name='value'))

,development_missing_rate,dtype,name,position,test_missing_rate
0,0.001789,float64,return_1d,0,0.000000
1,0.001789,float64,momentum_5d,1,0.000000
2,0.003578,float64,momentum_21d,2,0.000000
3,0.042934,float64,momentum_63d,3,0.000000
4,0.085868,float64,momentum_126d,4,0.000000
...,...,...,...,...,...
60,0.000000,float64,topic_capital_allocation_share,60,0.000000
61,0.019678,float64,topic_risk_regulation_sentiment,61,0.006623
62,0.000000,float64,topic_risk_regulation_share,62,0.000000
63,0.295170,float64,topic_technology_ai_sentiment,63,0.205298


development    7c7c72ec56e1dde8674842d76244b55b0f0cce2869f97a...
test           673198d8bb243f18f0603554a27ccc7671fdf637d40200...
Name: sha256, dtype: str

first_feature_date    2021-08-25T00:00:00
last_feature_date     2025-05-09T00:00:00
test_start            2025-07-01T00:00:00
validation_start      2024-07-01T00:00:00
Name: value, dtype: str

## Policy-aware loading

A rejected model requires an explicit research override. Production inference will request `champion` and therefore fail closed when no model passes the gates.

In [7]:
registered = load_registered_model(REGISTRY, version='latest', allow_rejected=True)
print('Loaded for research:', registered.version)
print('Boosted rounds:', registered.model.get_booster().num_boosted_rounds())
print('Features:', registered.model.get_booster().num_features())

Loaded for research: xgboost-20260917T064713Z-7c7c72ec
Boosted rounds: 69
Features: 65


In [8]:
display(Markdown((version_directory / 'model_card.md').read_text(encoding='utf-8')))

# AlphaLens Model Card: xgboost-20260917T064713Z-7c7c72ec

## Purpose

Predict 30-trading-session stock excess return versus SPY after an 
earnings call or SEC 10-K/10-Q event.

## Status

**REJECTED**

## Test Metrics

- MAE: 0.063018
- RMSE: 0.083351
- Directional accuracy: 52.318%
- Spearman IC: -0.105710

## Net Backtest

- Total return: -5.178%
- Sharpe: -0.4074
- Maximum drawdown: -11.940%

## Limitations

- The universe contains only 20 large US equities.
- The untouched test period is approximately one year.
- SHAP explanations describe model behavior, not causality.
- Rejected models must not be served as production predictions.
- This research artifact is not investment advice.
